# 05 - Exploratory Data Analysis (EDA)

Explore Shopsy kitchen product prices, discounts, ratings, reviews, materials, and capacities.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("shopsy_kitchen_products.csv")

print("Dataset loaded successfully!")
print("=" * 60)
print("\nFirst 5 rows:")
print(df.head())
print("\nLast 5 rows:")
print(df.tail())
print("\nDataset Shape:")
print(df.shape)
print("\nColumn Names:")
print(df.columns.tolist())
print("\nData Types:")
print(df.dtypes)
print("\nDataset Information:")
df.info()
print("\nStatistical Summary:")
print(df.describe())
print("\nMissing Values:")
print(df.isnull().sum())
print("\nMissing Value Percentage:")
print((df.isnull().sum() / len(df)) * 100)
print("\nDuplicate Rows:")
print(df.duplicated().sum())
print("\nUnique Values:")
for column in df.columns:
    print(column, ":", df[column].nunique())

## Data Cleaning

Convert price, discount, rating, and review fields into numeric values for analysis.

In [ ]:
df["Price_INR"] = (
    df["Price"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("â‚¹", "", regex=False)
    .str.replace(",", "", regex=False)
)
df["Price_INR"] = pd.to_numeric(df["Price_INR"], errors="coerce")

df["Discount_Pct"] = (
    df["Discount"].astype(str).str.replace("%", "", regex=False)
)
df["Discount_Pct"] = pd.to_numeric(df["Discount_Pct"], errors="coerce")
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
df["Reviews"] = pd.to_numeric(df["Reviews"], errors="coerce")
df["Reviews"] = df["Reviews"].fillna(df["Reviews"].median())

print("Cleaned Dataset:")
print(df[["Product_Name", "Price_INR", "Discount_Pct", "Rating", "Reviews"]].head())

## Univariate Analysis

Review the individual distributions of price, discount, rating, and reviews.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(df["Price_INR"].dropna(), bins=10)
axes[0, 0].set_title("Price Distribution")
axes[0, 0].set_xlabel("Price (INR)")
axes[0, 0].set_ylabel("Number of Products")

axes[0, 1].hist(df["Discount_Pct"].dropna(), bins=10)
axes[0, 1].set_title("Discount Distribution")
axes[0, 1].set_xlabel("Discount (%)")
axes[0, 1].set_ylabel("Number of Products")

axes[1, 0].hist(df["Rating"].dropna(), bins=10)
axes[1, 0].set_title("Rating Distribution")
axes[1, 0].set_xlabel("Rating")
axes[1, 0].set_ylabel("Number of Products")

axes[1, 1].hist(df["Reviews"].dropna(), bins=10)
axes[1, 1].set_title("Reviews Distribution")
axes[1, 1].set_xlabel("Number of Reviews")
axes[1, 1].set_ylabel("Number of Products")

plt.tight_layout()
plt.show()

## Categorical and Bivariate Analysis

Compare product materials and examine relationships between price, discount, rating, and reviews.

In [ ]:
print("Material Distribution:")
print(df["Material"].value_counts())

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
df["Material"].value_counts().plot(kind="bar", ax=axes[0, 0])
axes[0, 0].set_title("Products by Material")
axes[0, 0].set_xlabel("Material")
axes[0, 0].set_ylabel("Number of Products")
axes[0, 0].tick_params(axis="x", rotation=45)

print("\nCapacity Distribution:")
print(df["Capacity"].value_counts().head(15))

plots = [("Discount_Pct", "Price_INR", "Price vs Discount", "Discount (%)", "Price (INR)"),
         ("Rating", "Price_INR", "Price vs Rating", "Rating", "Price (INR)"),
         ("Reviews", "Price_INR", "Price vs Reviews", "Reviews", "Price (INR)"),
         ("Rating", "Reviews", "Rating vs Reviews", "Rating", "Reviews")]
for axis, (x_column, y_column, title, x_label, y_label) in zip(axes.flat[1:], plots):
    axis.scatter(df[x_column], df[y_column])
    axis.set_title(title)
    axis.set_xlabel(x_label)
    axis.set_ylabel(y_label)

plt.tight_layout()
plt.show()

## Grouped Analysis and Top Products

In [ ]:
avg_price_material = df.groupby("Material")["Price_INR"].mean().sort_values(ascending=False)
avg_rating_material = df.groupby("Material")["Rating"].mean().sort_values(ascending=False)
avg_discount_material = df.groupby("Material")["Discount_Pct"].mean().sort_values(ascending=False)

print("Average Price by Material:")
print(avg_price_material)
print("\nAverage Rating by Material:")
print(avg_rating_material)
print("\nAverage Discount by Material:")
print(avg_discount_material)

top_reviews = df.sort_values("Reviews", ascending=False).head(10)
top_rated = df.sort_values(["Rating", "Reviews"], ascending=[False, False]).head(10)

print("\nTop 10 Most Reviewed Products:")
print(top_reviews[["Product_Name", "Price_INR", "Discount_Pct", "Rating", "Reviews"]])
print("\nTop 10 Rated Products:")
print(top_rated[["Product_Name", "Price_INR", "Rating", "Reviews"]])

## Correlation, Outliers, and Business Questions

In [ ]:
numeric_columns = ["Price_INR", "Discount_Pct", "Rating", "Reviews"]
correlation = df[numeric_columns].corr()
print("Correlation Matrix:")
print(correlation)

plt.figure(figsize=(8, 6))
plt.imshow(correlation, interpolation="nearest")
plt.colorbar()
plt.xticks(range(len(correlation.columns)), correlation.columns, rotation=45)
plt.yticks(range(len(correlation.columns)), correlation.columns)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].boxplot(df["Price_INR"].dropna())
axes[0].set_title("Price Outlier Analysis")
axes[0].set_ylabel("Price (INR)")
axes[1].boxplot(df["Reviews"].dropna())
axes[1].set_title("Reviews Outlier Analysis")
axes[1].set_ylabel("Number of Reviews")
plt.tight_layout()
plt.show()

cheapest = df.loc[df["Price_INR"].idxmin()]
expensive = df.loc[df["Price_INR"].idxmax()]
highest_discount = df.loc[df["Discount_Pct"].idxmax()]
most_reviewed = df.loc[df["Reviews"].idxmax()]
highest_rated = df.loc[df["Rating"].idxmax()]

print("Cheapest Product:", cheapest["Product_Name"], "| Price:", cheapest["Price_INR"])
print("Most Expensive Product:", expensive["Product_Name"], "| Price:", expensive["Price_INR"])
print("Highest Discount Product:", highest_discount["Product_Name"], "| Discount:", highest_discount["Discount_Pct"], "%")
print("Most Reviewed Product:", most_reviewed["Product_Name"], "| Reviews:", most_reviewed["Reviews"])
print("Highest Rated Product:", highest_rated["Product_Name"], "| Rating:", highest_rated["Rating"])

print("\nFinal Dataset Shape:")
print(df.shape)
print("Final Columns:")
print(df.columns.tolist())
print("EDA COMPLETED SUCCESSFULLY!")